# 🧠 EEG Pipeline — 纯 CPU 多核并发加速节点 (Google Colab High-RAM)

本 Notebook 专为在 **Google Colab High-RAM (53GB RAM) 纯 CPU 实例** 上运行设计，**无需任何 GPU 或显存**，核心特性：
- **零配额冲突**：可与您的 GPU 节点或主号会话**同时并行运行**，完全不挤占 Google Colab 的 GPU 配额；
- **独立被试批次**：本节点挂载 `eeg_data_colab_cpu.tar.gz` 数据包（包含 Sub 2~9、17~28 共 20 个被试，与前序 GPU 脚本中的 Sub 12~16 严格正交不重叠）；
- **多核并发调优**：释放 8 vCPUs 与 53GB 内存，告别本地极端保守的内存限制；
- **统一云端持久化**：每跑完一个被试，产物立即自动增量同步至 Google Drive 的同一个 `output_data` 目录！

## 1. 硬件规格与运行时设置
运行前请确保：在顶部菜单栏依次点击 **Runtime (代码执行程序) -> Change runtime type (更改运行时类型)**：
- **Hardware accelerator (硬件加速器)**: 选择 **None (纯 CPU)**
- **Runtime shape (运行时配置)**: 选择 **High-RAM (大内存，~53GB)**

In [ ]:
import os, sys, psutil

print("=" * 60)
print("🖥️  Colab CPU 硬件与物理内存检测")
print("=" * 60)

cpu_count = os.cpu_count()
vm = psutil.virtual_memory()
total_ram_gb = vm.total / (1024 ** 3)
avail_ram_gb = vm.available / (1024 ** 3)
print(f"CPU 核心数: {cpu_count} vCPUs")
print(f"系统总内存: {total_ram_gb:.2f} GB (可用: {avail_ram_gb:.2f} GB)")
if total_ram_gb < 30:
    print("⚠️ 提示: 当前为标准内存实例。如果可以，建议在 Runtime 设置中切换为 High-RAM 获得 53GB 充裕空间！")
else:
    print("✅ High-RAM 53GB 超大内存已就绪！")

## 2. 挂载 Google Drive (连接 5TB 云存储)

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# 定义 Google Drive 上的持久化工作目录
DRIVE_WORKSPACE = "/content/drive/MyDrive/EEG_Pipeline"
DRIVE_DATA_ARCHIVE = f"{DRIVE_WORKSPACE}/eeg_data_colab_cpu.tar.gz"
DRIVE_OUTPUT_DIR = f"{DRIVE_WORKSPACE}/output_data"

os.makedirs(DRIVE_WORKSPACE, exist_ok=True)
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
print(f"✅ Google Drive 工作目录就绪: {DRIVE_WORKSPACE}")
print(f"📦 数据压缩包路径: {DRIVE_DATA_ARCHIVE}")
print(f"📁 结果输出目录: {DRIVE_OUTPUT_DIR}")

## 3. 克隆代码仓库并安装纯 CPU 依赖
无需安装 CuPy 或 CUDA 驱动，仅安装所需的信号处理与数值计算库。

In [ ]:
REPO_DIR = "/content/eeg-nonlinear-pipeline"

if not os.path.exists(REPO_DIR):
    print("Cloning repository...")
    !git clone https://github.com/TH-yi/eeg-nonlinear-pipeline.git {REPO_DIR}
else:
    print("Repository already exists, pulling latest updates...")
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}

# 安装纯 CPU 计算所需的依赖 (免装 cupy，秒速就绪)
!pip install -q typer mat73 nolds antropy mne psutil scipy rich pyrqa
print("✅ 纯 CPU 依赖库安装配置完成！")

## 4. 解压 CPU 批次数据至本地 NVMe 高速 SSD
解压 `eeg_data_colab_cpu.tar.gz` 到 `/content/local_data`，并自动拉取 Drive 已完成的被试结果以便断点跳过。

In [ ]:
LOCAL_DATA_DIR = "/content/local_data"
LOCAL_OUTPUT_DIR = "/content/local_output"
LOCAL_CACHE_DIR = "/content/sig_cache"

os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
os.makedirs(LOCAL_OUTPUT_DIR, exist_ok=True)
os.makedirs(LOCAL_CACHE_DIR, exist_ok=True)

# 检查本地数据是否已存在，不存在则从 Drive 解压
local_files = [f for f in os.listdir(LOCAL_DATA_DIR) if f.endswith('.mat')]
if len(local_files) == 0:
    if os.path.exists(DRIVE_DATA_ARCHIVE):
        print(f"📦 正在从 Google Drive 解压数据包: {DRIVE_DATA_ARCHIVE} -> {LOCAL_DATA_DIR} ...")
        !tar -xzf {DRIVE_DATA_ARCHIVE} -C {LOCAL_DATA_DIR}
        mats = [f for f in os.listdir(LOCAL_DATA_DIR) if f.endswith('.mat')]
        print(f"✅ 解压完成！共发现 {len(mats)} 个独立 Subject .mat 文件在本地高速盘")
    else:
        print(f"⚠️ 未在 Google Drive 找到压缩包: {DRIVE_DATA_ARCHIVE}")
else:
    print(f"✅ 本地高速盘已有 {len(local_files)} 个 Subject 数据，直接复用无需再次解压！")

# 同步 Google Drive 已有的输出产物到本地 (避免重复计算)
if os.path.exists(DRIVE_OUTPUT_DIR):
    !cp -n {DRIVE_OUTPUT_DIR}/*_features.json {LOCAL_OUTPUT_DIR}/ 2>/dev/null || true
    existing_done = len([f for f in os.listdir(LOCAL_OUTPUT_DIR) if f.endswith('_features.json')])
    print(f"ℹ️ 已从 Google Drive 同步已完成的 {existing_done} 个被试结果用于断点跳过。")

## 5. 配置纯 CPU 高并发资源并启动流水线
可灵活选择计算模式：
- `METHOD = 'rqa'`: 递归分析 (纯 CPU 多核并发)
- `METHOD = 'nonlinear'`: 非线性动力学特征 (Sample/Fuzzy Entropy, HFD 等)

可通过 `SUBJECTS_FILTER` 进一步指定只跑某些特定被试（如 `"2,3,4,5"`），留空则自动处理本地全部未完成被试。

In [ ]:
import os, subprocess, time, glob, shutil, sys

# ================== 运行参数配置 ==================
METHOD = "rqa"          # "rqa" 或 "nonlinear"
SKIP_EXISTING = True    # 开启断点续跑（跳过已处理被试）
SUBJECTS_FILTER = ""    # 可选: 指定被试过滤 (如 "2,3,4,5")，留空则跑本地全部未完成被试
# ==================================================

# 环境变量注入 (最大化调动 CPU 8 vCPUs 算力与 53GB 内存)
os.environ["EEG_MEM_LIMIT"] = "0.7"           # 允许使用 70% 内存安全预算
os.environ["EEG_CPU_RATIO"] = "0.85"          # 调动 85% CPU 核心
os.environ["EEG_PARALLEL_TASKS"] = "4"        # 通道级并发数
os.environ["EEG_CACHE_DIR"] = LOCAL_CACHE_DIR

local_mats = [f for f in os.listdir(LOCAL_DATA_DIR) if f.endswith('.mat')]
if len(local_mats) == 0:
    raise RuntimeError(f"❌ 错误: {LOCAL_DATA_DIR} 中未找到任何被试 .mat 文件！请先等待云端数据包上传完成，并重新运行步骤 4 解压！")
print(f"📦 本地高速盘就绪: 发现 {len(local_mats)} 个待处理 Subject 数据。")

print(f"🚀 启动 EEG 纯 CPU 分析流水线: Method={METHOD} | GPU=False | SkipExisting={SKIP_EXISTING}")

# 构造启动指令
cmd = [
    sys.executable, "main.py", "process",
    "--data-dir", LOCAL_DATA_DIR,
    "--output-dir", LOCAL_OUTPUT_DIR,
    "--method", METHOD,
    "--no-use-gpu",
    "--skip-existing" if SKIP_EXISTING else "--force"
]
if SUBJECTS_FILTER.strip():
    cmd.extend(["--subjects", SUBJECTS_FILTER.strip()])

start_time = time.time()
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
synced_files = set(os.listdir(DRIVE_OUTPUT_DIR))

try:
    for line in proc.stdout:
        print(line, end="")
        # 检查是否有新产出的 features.json 并立即同步至 Drive (增量安全备份)
        current_files = set(glob.glob(f"{LOCAL_OUTPUT_DIR}/*_features.json") + glob.glob(f"{LOCAL_OUTPUT_DIR}/*_NL_Results.mat"))
        for fpath in current_files:
            fname = os.path.basename(fpath)
            if fname not in synced_files:
                shutil.copy2(fpath, os.path.join(DRIVE_OUTPUT_DIR, fname))
                synced_files.add(fname)
                print(f"[☁️ 云端备份] 成功同步 {fname} 至 Google Drive!")
except KeyboardInterrupt:
    print("⚠️ 收到中断信号，正在安全停止...")
    proc.terminate()

proc.wait()
elapsed = time.time() - start_time

# 完整同步所有汇总文件
!cp -r {LOCAL_OUTPUT_DIR}/* {DRIVE_OUTPUT_DIR}/

print("=" * 60)
print(f"🎉 全部处理与同步完成！总耗时: {elapsed / 60:.2f} 分钟")
print(f"📁 结果已持久化保存至 Google Drive: {DRIVE_OUTPUT_DIR}")
print("=" * 60)


## 6. 全局进度与特征汇总检验

In [ ]:
import json, glob, os

results = sorted(glob.glob(f"{DRIVE_OUTPUT_DIR}/*_features.json"))
print(f"📊 Google Drive 中已完成特征提取的 Subject 总数: {len(results)} / 28")
for r in results:
    print(f"  ✅ {os.path.basename(r)}")

# 汇总进度百分比
pct = (len(results) / 28) * 100
print(f"\n🔥 全局总完成进度: {pct:.1f}%")